In [ ]:
import torchvision
import torch
import torch.nn as nn

from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, length=1000):
        super().__init__()
        self.length = length
        self.minus = np.array([[0, 0, 0, 0, 0],
                  [0, 0, 0, 0, 0],
                  [1, 1, 1, 1, 1],
                  [0, 0, 0, 0, 0],
                  [0, 0, 0, 0, 0]])
        self.plus = np.array([[0, 0, 1, 0, 0],
                 [0, 0, 1, 0, 0],
                 [1, 1, 1, 1, 1],
                 [0, 0, 1, 0, 0],
                 [0, 0, 1, 0, 0]])
        self.equal = np.array([[0, 0, 0, 0, 0],
                  [1, 1, 1, 1, 1],
                  [0, 0, 0, 0, 0],
                  [1, 1, 1, 1, 1],
                  [0, 0, 0, 0, 0]])
        self.multiply = np.array([[1, 0, 0, 0, 1],
                  [0, 1, 0, 1, 0],
                  [0, 0, 1, 0, 0],
                  [0, 1, 0, 1, 0],
                  [1, 0, 0, 0, 1]])
        self.divide = np.array([[1, 0, 0, 0, 0],
                  [0, 1, 0, 0, 0],
                  [0, 0, 1, 0, 0],
                  [0, 0, 0, 1, 0],
                  [0, 0, 0, 0, 1]])

        self.dataset, self.labels = self.create_dataset()

    def add_noise(self, array, noise_level):
        noise = np.random.random(array.shape) < noise_level
        return np.logical_xor(array, noise).astype(float)

    def create_dataset_with_noise(self, data, length, noise_lvl):
        return np.array([self.add_noise(data, noise_lvl) for _ in range(length)])

    def create_dataset(self):
        dataset = []
        labels = []
        noise_lvl = 0.1
        position = 0
        for i in (self.minus, self.plus, self.equal, self.multiply, self.divide):
            label = [0, 0, 0, 0, 0]
            dataset.extend(self.create_dataset_with_noise(i, self.length, noise_lvl))
            label[position] = 1
            labels.extend([label] * self.length)
            position += 1
        return torch.tensor(np.array(dataset)).float(), torch.tensor(np.array(labels)).float()

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        return (self.dataset[index], self.labels[index])







In [ ]:
dataset = Dataset()
print(len(dataset))
print(dataset[200])

In [ ]:
train_data = Dataset()
test_data = Dataset(100)

In [ ]:
batch_size = 20
train_dataloader =torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, drop_last=True)
test_dataloader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=True)


In [ ]:
class MPL(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(25, 10)
        self.layer2 = nn.Linear(10, 5)
        self.activation = nn.Sigmoid()
        self.flatten = nn.Flatten()

    def forward(self, x):
        x = self.flatten(x)
        x = self.activation(self.layer1(x))
        x = self.layer2(x)
        return x



In [ ]:
model = MPL()
loss = nn.CrossEntropyLoss()
lr = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=lr)


In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer):
    model.train()
    for data, label in tqdm(dataloader):
        optimizer.zero_grad()
        output = model(data)
        loss = loss_fn(output, label)
        loss.backward()
        optimizer.step()


def test_loop(dataloader, model, loss_fn, optimizer):
    model.eval()
    correct = 0
    test_loss = 0
    with torch.no_grad():
        for data, label in dataloader:
            output = model(data)
            test_loss += loss_fn(output, label).item()
            correct += (torch.argmax(output, dim=1) == torch.argmax(label, dim=1)).float().sum().item()
        test_loss /= len(dataloader)
        correct /= len(dataloader.dataset)
    print(f'test loss:{test_loss:.2f}, test accuracy:{correct * 100:.2f}%')
    return test_loss, correct



In [ ]:
epochs = 50
best_loss = float('inf')
counter = 0
n = 10
for epoch in range(epochs):
    train_loop(train_dataloader, model, loss, optimizer)
    test_loss = test_loop(test_dataloader, model, loss, optimizer)
    test_loss, accuracy = np.round(test_loss, decimals=1)
    if test_loss >= best_loss:
        counter += 1
    else:
        counter = 0
        best_loss = test_loss
    if counter >= n:
        print(f'I DON\'T WANT STUDY MORE, Accuracy is {accuracy*100}%')
        break

In [ ]:
test_img = np.array([[1, 0, 0, 0, 1],
                  [0, 1, 0, 1, 0],
                  [1, 0, 1, 0, 1],
                  [0, 1, 0, 1, 0],
                  [1, 0, 0, 0, 1]])

categories = ['plus', 'minus', 'equal', 'multiply', 'divide']

img_for_model = torch.Tensor(test_img).unsqueeze(0).unsqueeze(0)

model.eval()

print(categories[model(img_for_model).argmax(1).item()])
